# Phase 2: Evaluation - Retrieval & Generate Predictions

Quy trình:
1. Load 200 query_id từ ground truth JSONL
2. Load tất cả models (TFIDF, Ingredient_TFIDF, Keyword, Hybrid)
3. Với mỗi query_id:
   - Tạo query representation theo từng method
   - Retrieve top K=10 (exclude self)
   - Ghi prediction
4. Lưu predictions vào `evaluation_jsonl/<method>_pred.jsonl`

In [11]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
import json
import os
from pathlib import Path
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
# Setup paths
DATA_PATH = r"E:\DS300-UIT-RecommenderSystem\Finalproject\data\all_recipes_final.csv"
MODELS_PATH = r"E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\Saved_models"
GROUND_TRUTH_PATH = r"E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\eval_ground_truth.jsonl"
OUTPUT_DIR = r"E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl"

# Create output directory if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Models path: {MODELS_PATH}")
print(f"Ground truth path: {GROUND_TRUTH_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

Data path: E:\DS300-UIT-RecommenderSystem\Finalproject\data\all_recipes_final.csv
Models path: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\Saved_models
Ground truth path: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\eval_ground_truth.jsonl
Output directory: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl


## Step 1: Load Ground Truth và Data

In [13]:
# Load ground truth JSONL and extract query_ids
ground_truth = []
with open(GROUND_TRUTH_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        ground_truth.append(json.loads(line.strip()))

query_ids = [item['query_id'] for item in ground_truth]
print(f"Loaded {len(query_ids)} query_ids from ground truth")
print(f"First 5 query_ids: {query_ids[:5]}")

Loaded 200 query_ids from ground truth
First 5 query_ids: [6302, 3779, 768, 3399, 4561]


In [14]:
# Load data (all recipes - 10k candidate pool)
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} recipes")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loaded 10263 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'ingredients_normalized', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


,title,type_of_food,link,description,ingredients,ingredients_normalized,step,note,num_of_ingredients,cook_time,num_of_people,calories,source
0,Cách muối dưa hành truyền thống,Món Tết,https://vnexpress.net/doi-song-cooking-cach-mu...,Dưa hành muối là món ăn truyền thống ngày Tết ...,"['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọ...","{'tro bếp hoặc nước vo gọa', 'đường', 'cà rốt ...",['Bước 1: Chọn hành củ: Nên chọn hành củ ta bá...,[],5,45 phút,8-10 người,459 kcal,vnexpress
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,https://vnexpress.net/doi-song-cooking-su-hao-...,Đĩa xào khô ráo với su hào giòn ngọt quyện với...,"['2 củ su hào non', '1 con mực khô', '1/2 củ c...","{'muối', 'mỡ lợn hoặc dầu ăn', 'đường', 'gia v...",['Bước 1: Chọn và sơ chế mực: Người dân làng g...,['Su hào xào mực cùng với canh măng mực là hai...,6,50 phút,4 - 5 người,1.162 kcal,vnexpress
2,Canh măng ngày Tết cổ truyền Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-canh-ma...,"Măng ngấu vị, giòn ngon, móng giò hầm vừa độ s...","['800 gr măng khô', '2 móng giò lợn', 'Nước dù...","{'muối', 'móng giò lợn', 'nước vo gạo ngâm măn...","['Bước 1: Chọn măng khô: Theo lối cũ, người nộ...",['Nếu tận dụng nước luộc gà nấu canh măng thì ...,6,100 phút,8 - 10 người,4.930 kcal,vnexpress
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-gia-han...,Đây là món ăn cổ truyền thường thấy trong cỗ T...,"['2 bộ lòng mề gà', '100 gr lạc', '50 gr hạt đ...","{'bộ lòng mề gà', 'muối', 'mỡ lợn', 'gia vị: m...",['Bước 1: Chọn và sơ chế lạc: Chọn lạc khô chắ...,['Hạnh nhân xào (hay giả hạnh nhân) là món ăn ...,8,60 phút,4-5 người,1.112 kcal,vnexpress
4,Chả bì ớt xiêm xanh,Món Tết,https://vnexpress.net/doi-song-cooking-cha-bi-...,"Chả bì bóng đẹp, gói đều tay. Khi ăn vị ngọt m...","['500 gr giò sống', '300 gr bì lợn', '20 - 30 ...","{'muối', 'gừng để sơ chế bì', 'bì lợn', 'hành ...","['Bước 1: Chọn và sơ chế bì lợn, chuẩn bị giò ...",['Nên sơ chế kỹ bì lợn để chả được thơm. Tùy t...,6,60 phút,5-6 người,2.512 kcal,vnexpress


In [16]:
# Add recipe_id column (using index as recipe_id)
df['recipe_id'] = df.index
print(f"Added recipe_id column. Shape: {df.shape}")
print(f"Recipe ID range: {df['recipe_id'].min()} to {df['recipe_id'].max()}")

Added recipe_id column. Shape: (10263, 14)
Recipe ID range: 0 to 10262


## Step 2: Load All Models

In [15]:
# Method 1: TFIDF (text-based: title + description + steps)
print("Loading TFIDF model...")
with open(os.path.join(MODELS_PATH, "TFIDF", "tfidf_vectorizer.pkl"), 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
    
tfidf_similarity = np.load(os.path.join(MODELS_PATH, "TFIDF", "tfidf_similarity.npy"))

with open(os.path.join(MODELS_PATH, "TFIDF", "tfidf_processed_data.pkl"), 'rb') as f:
    tfidf_data = pickle.load(f)

print(f"TFIDF similarity matrix shape: {tfidf_similarity.shape}")
print(f"TFIDF data shape: {len(tfidf_data)}")

Loading TFIDF model...
TFIDF similarity matrix shape: (10263, 10263)
TFIDF data shape: 10263


In [17]:
# Method 2: Ingredient_TFIDF (ingredient-based)
print("Loading Ingredient_TFIDF model...")
with open(os.path.join(MODELS_PATH, "Ingredient_TFIDF", "ingredient_tfidf_vectorizer.pkl"), 'rb') as f:
    ingredient_tfidf_vectorizer = pickle.load(f)
    
ingredient_tfidf_similarity = np.load(os.path.join(MODELS_PATH, "Ingredient_TFIDF", "ingredient_tfidf_similarity.npy"))

print(f"Ingredient TFIDF similarity matrix shape: {ingredient_tfidf_similarity.shape}")

Loading Ingredient_TFIDF model...
Ingredient TFIDF similarity matrix shape: (10263, 10263)


In [18]:
# Method 3: Keyword (TF-IDF on keywords extracted from title)
print("Loading Keyword model...")
keyword_similarity = np.load(os.path.join(MODELS_PATH, "Keyword", "keyword_similarity.npy"))

print(f"Keyword similarity matrix shape: {keyword_similarity.shape}")

Loading Keyword model...
Keyword similarity matrix shape: (10263, 10263)


In [19]:
# Method 4: Hybrid (combine text_tfidf + ingredient_tfidf)
print("Loading Hybrid model...")
hybrid_similarity = np.load(os.path.join(MODELS_PATH, "Hybrid", "hybrid_similarity.npy"))

print(f"Hybrid similarity matrix shape: {hybrid_similarity.shape}")

Loading Hybrid model...
Hybrid similarity matrix shape: (10263, 10263)


## Step 3: Implement Retrieval Functions

In [20]:
def retrieve_top_k(query_idx, similarity_matrix, k=10, exclude_self=True):
    """
    Retrieve top K similar items for a query
    
    Args:
        query_idx: index of query item in dataframe
        similarity_matrix: precomputed similarity matrix
        k: number of items to retrieve
        exclude_self: whether to exclude query item itself
    
    Returns:
        list of (doc_id, score) tuples
    """
    # Get similarity scores for query
    scores = similarity_matrix[query_idx].copy()
    
    # Exclude self if requested
    if exclude_self:
        scores[query_idx] = -1  # Set to -1 to exclude from top-k
    
    # Get top K indices
    top_k_indices = np.argsort(scores)[::-1][:k]
    
    # Get doc_ids and scores
    results = []
    for idx in top_k_indices:
        doc_id = df.iloc[idx]['recipe_id']
        score = float(scores[idx])
        results.append({"doc_id": int(doc_id), "score": score})
    
    return results

# Test function
test_query_idx = 0
test_results = retrieve_top_k(test_query_idx, tfidf_similarity, k=5)
print(f"Test retrieval for index {test_query_idx}:")
print(f"Query recipe_id: {df.iloc[test_query_idx]['recipe_id']}")
print(f"Top 5 results: {test_results}")

Test retrieval for index 0:
Query recipe_id: 0
Top 5 results: [{'doc_id': 52, 'score': 0.6395764849733627}, {'doc_id': 9321, 'score': 0.4543578750013707}, {'doc_id': 10057, 'score': 0.3689254056400364}, {'doc_id': 8290, 'score': 0.36855233891334216}, {'doc_id': 619, 'score': 0.33020255854014563}]


## Step 4: Run Evaluation Loop for All Methods

In [21]:
def run_evaluation(method_name, similarity_matrix, query_ids, df, k=10):
    """
    Run evaluation for a specific method
    
    Args:
        method_name: name of the method (for output file)
        similarity_matrix: precomputed similarity matrix
        query_ids: list of query recipe_ids
        df: dataframe containing all recipes
        k: number of items to retrieve
    
    Returns:
        list of predictions (one per query)
    """
    predictions = []
    
    # Create recipe_id to index mapping
    recipe_id_to_idx = {recipe_id: idx for idx, recipe_id in enumerate(df['recipe_id'])}
    
    for query_id in tqdm(query_ids, desc=f"Evaluating {method_name}"):
        # Get query index
        query_idx = recipe_id_to_idx[query_id]
        
        # Retrieve top K (excluding self)
        top_k = retrieve_top_k(query_idx, similarity_matrix, k=k, exclude_self=True)
        
        # Create prediction record
        pred_record = {
            "query_id": int(query_id),
            "top10": top_k
        }
        predictions.append(pred_record)
    
    return predictions

# Test with small subset first
print("Testing with first 5 queries...")
test_predictions = run_evaluation("test", tfidf_similarity, query_ids[:5], df, k=10)
print(f"\nExample prediction:")
print(json.dumps(test_predictions[0], indent=2, ensure_ascii=False))

Testing with first 5 queries...


Evaluating test: 100%|██████████| 5/5 [00:00<00:00, 595.36it/s]


Example prediction:
{
  "query_id": 6302,
  "top10": [
    {
      "doc_id": 6547,
      "score": 0.7862034845555061
    },
    {
      "doc_id": 6605,
      "score": 0.7484998247435155
    },
    {
      "doc_id": 6160,
      "score": 0.7422538990467994
    },
    {
      "doc_id": 7121,
      "score": 0.6669771498939066
    },
    {
      "doc_id": 4790,
      "score": 0.6642594310179872
    },
    {
      "doc_id": 6264,
      "score": 0.6633940945413028
    },
    {
      "doc_id": 6305,
      "score": 0.6628790498421143
    },
    {
      "doc_id": 6319,
      "score": 0.6604463496648965
    },
    {
      "doc_id": 6290,
      "score": 0.6572526836002297
    },
    {
      "doc_id": 6568,
      "score": 0.6532169015263641
    }
  ]
}


In [22]:
# Run evaluation for all 4 methods
K = 10  # Top K to retrieve

methods = [
    ("TFIDF", tfidf_similarity),
    ("Ingredient_TFIDF", ingredient_tfidf_similarity),
    ("Keyword", keyword_similarity),
    ("Hybrid", hybrid_similarity)
]

all_predictions = {}

for method_name, similarity_matrix in methods:
    print(f"\n{'='*60}")
    print(f"Running evaluation for {method_name}...")
    print(f"{'='*60}")
    
    # Run evaluation
    predictions = run_evaluation(method_name, similarity_matrix, query_ids, df, k=K)
    all_predictions[method_name] = predictions
    
    # Save to JSONL file
    output_file = os.path.join(OUTPUT_DIR, f"{method_name}_pred.jsonl")
    with open(output_file, 'w', encoding='utf-8') as f:
        for pred in predictions:
            f.write(json.dumps(pred, ensure_ascii=False) + '\n')
    
    print(f"✓ Saved {len(predictions)} predictions to {output_file}")

print(f"\n{'='*60}")
print("All evaluations completed!")
print(f"{'='*60}")


Running evaluation for TFIDF...


Evaluating TFIDF: 100%|██████████| 200/200 [00:00<00:00, 1070.26it/s]


✓ Saved 200 predictions to E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl\TFIDF_pred.jsonl

Running evaluation for Ingredient_TFIDF...


Evaluating Ingredient_TFIDF: 100%|██████████| 200/200 [00:00<00:00, 1216.84it/s]


✓ Saved 200 predictions to E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl\Ingredient_TFIDF_pred.jsonl

Running evaluation for Keyword...


Evaluating Keyword: 100%|██████████| 200/200 [00:00<00:00, 931.53it/s]


✓ Saved 200 predictions to E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl\Keyword_pred.jsonl

Running evaluation for Hybrid...


Evaluating Hybrid: 100%|██████████| 200/200 [00:00<00:00, 1125.51it/s]

✓ Saved 200 predictions to E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl\Hybrid_pred.jsonl

All evaluations completed!


## Step 5: Verify Output Files

In [23]:
# Verify generated files
import os

print("Generated prediction files:")
print("="*60)
for filename in sorted(os.listdir(OUTPUT_DIR)):
    if filename.endswith('_pred.jsonl'):
        filepath = os.path.join(OUTPUT_DIR, filename)
        file_size = os.path.getsize(filepath)
        
        # Count lines
        with open(filepath, 'r', encoding='utf-8') as f:
            num_lines = sum(1 for _ in f)
        
        print(f"✓ {filename:30s} - {num_lines} predictions ({file_size:,} bytes)")

print("="*60)

Generated prediction files:
✓ Hybrid_pred.jsonl              - 200 predictions (100,667 bytes)
✓ Ingredient_TFIDF_pred.jsonl    - 200 predictions (100,463 bytes)
✓ Keyword_pred.jsonl             - 200 predictions (95,248 bytes)
✓ TFIDF_pred.jsonl               - 200 predictions (100,158 bytes)


In [24]:
# Show sample predictions from each method
for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid"]:
    filepath = os.path.join(OUTPUT_DIR, f"{method_name}_pred.jsonl")
    
    print(f"\n{'='*60}")
    print(f"Sample from {method_name}:")
    print('='*60)
    
    with open(filepath, 'r', encoding='utf-8') as f:
        first_pred = json.loads(f.readline())
        
    print(f"Query ID: {first_pred['query_id']}")
    print(f"Top 10 predictions:")
    for i, item in enumerate(first_pred['top10'], 1):
        # Get doc title
        doc_row = df[df['recipe_id'] == item['doc_id']].iloc[0]
        doc_title = doc_row['title']
        print(f"  {i}. Doc {item['doc_id']} (score: {item['score']:.4f}) - {doc_title[:60]}...")


Sample from TFIDF:
Query ID: 6302
Top 10 predictions:
  1. Doc 6547 (score: 0.7862) - Cá rô kho tương hột đậm đà thơm ngon dễ làm tại nhà...
  2. Doc 6605 (score: 0.7485) - Cá rô kho nghệ thơm lừng mềm ngon đậm đà hương vị...
  3. Doc 6160 (score: 0.7423) - Cá đù kho ngon miệng đậm đà hương vị cho bữa cơm cả nhà...
  4. Doc 7121 (score: 0.6670) - Cá rô rang muối dân dã lạ vị đổi món cho bữa cơm gia đình...
  5. Doc 4790 (score: 0.6643) - Cá rô nướng thơm ngon hấp dẫn đơn giản dễ làm cho bữa cơm...
  6. Doc 6264 (score: 0.6634) - Cá rô kho tộ thơm ngon đậm đà hấp dẫn cực đưa cơm...
  7. Doc 6305 (score: 0.6629) - Cá thu kho gừng cay cay ngon miệng đậm đà dễ làm...
  8. Doc 6319 (score: 0.6604) - Cá ngát kho gừng thơm ngon, đậm đà cho bữa cơm thêm tròn vị...
  9. Doc 6290 (score: 0.6573) - Món cá diếc kho tiêu thơm ngon đậm đà hấp dẫn tại nhà...
  10. Doc 6568 (score: 0.6532) - Cá bống kho tiêu ngon đậm vị, dân dã ai cũng mê tít...

Sample from Ingredient_TFIDF:
Query ID: 6302
Top 10 pr

## Step 6: Sanity Check - Verify No Self-Recommendations

In [25]:
# Verify that no prediction contains the query_id itself
def verify_no_self_recommendations(predictions):
    """Check if any prediction contains self-recommendation"""
    violations = []
    
    for pred in predictions:
        query_id = pred['query_id']
        doc_ids = [item['doc_id'] for item in pred['top10']]
        
        if query_id in doc_ids:
            violations.append(query_id)
    
    return violations

print("Checking for self-recommendations...")
print("="*60)

for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid"]:
    preds = all_predictions[method_name]
    violations = verify_no_self_recommendations(preds)
    
    if violations:
        print(f"❌ {method_name}: Found {len(violations)} self-recommendations!")
        print(f"   Query IDs with self-recommendation: {violations[:5]}...")
    else:
        print(f"✓ {method_name}: No self-recommendations found ({len(preds)} queries checked)")

print("="*60)

Checking for self-recommendations...
✓ TFIDF: No self-recommendations found (200 queries checked)
✓ Ingredient_TFIDF: No self-recommendations found (200 queries checked)
✓ Keyword: No self-recommendations found (200 queries checked)
✓ Hybrid: No self-recommendations found (200 queries checked)


## Summary

✅ **Phase 2 Evaluation Completed Successfully!**

**Quy trình đã thực hiện:**

1. ✓ Load 200 query_ids từ ground truth JSONL
2. ✓ Load all_recipes_final.csv (10,263 recipes - candidate pool)
3. ✓ Load 4 models:
   - TFIDF (text-based: title + description + steps)
   - Ingredient_TFIDF (ingredient-based)
   - Keyword (TF-IDF trên keywords từ title)
   - Hybrid (combine text_tfidf + ingredient_tfidf)

4. ✓ Với mỗi query_id (200 queries):
   - Xác định query item trong df
   - Retrieve top K=10 (exclude self)
   - Ghi prediction với doc_id và score

5. ✓ Lưu predictions vào 4 files JSONL:
   - `evaluation_jsonl/TFIDF_pred.jsonl`
   - `evaluation_jsonl/Ingredient_TFIDF_pred.jsonl`
   - `evaluation_jsonl/Keyword_pred.jsonl`
   - `evaluation_jsonl/Hybrid_pred.jsonl`

**Các ràng buộc đã đảm bảo:**
- ✓ Exclude self: không recommend chính query_id
- ✓ Candidate pool là toàn bộ 10k (trừ self)
- ✓ Không dùng Jaccard/rel trong Phase 2
- ✓ Format output đúng chuẩn JSONL

**Sẵn sàng cho Phase 3: Tính metrics (Precision, Recall, MRR, NDCG, ...)**